# EcoPlots Python Library — Samples Mode Demonstration

Welcome to the **EcoPlots Samples** demonstration notebook. This notebook covers every feature available in **samples mode** — from discovery and filtering, through to IGSN browsing, image viewing, and data export.

## About Samples Mode

In `samples` mode the EcoPlots client retrieves **physical specimens** collected during TERN Ecosystem Surveillance field surveys:

| Sample Type | Description |
|---|---|
| **Soil Pit Sample** | Bulk soil from a defined pit depth |
| **Soil Subsite Sample** | Replicate soil from sub-locations within a site |
| **Soil Metagenomic Sample** | DNA-grade soil for metagenomic analysis |
| **Plant Tissue Sample** | Subsampled tissue from a collected plant |
| **Plant Voucher Specimen** | Pressed herbarium voucher (may include photos) |

</br>

> **Note:** The *TERN Ecosystem Surveillance* dataset is applied automatically and cannot be removed.

</br>

---

## Table of Contents

1. [Setup & Initialisation](#1-setup--initialisation)
2. [Discovering Available Data](#2-discovering-available-data)
3. [Applying Filters](#3-applying-filters)
4. [Inspecting & Managing Filters](#4-inspecting--managing-filters)
5. [Summary & Preview](#5-summary--preview)
6. [Retrieving Data](#6-retrieving-data)
7. [IGSN Identifiers](#7-igsn-identifiers)
8. [Soil Analysis](#8-soil-analysis)
9. [Species Distribution](#9-species-distribution)
10. [Sample Image Viewer](#10-sample-image-viewer)
11. [Spatial Filter Widget](#11-spatial-filter-widget)

---

<a id="1-setup--initialisation"></a>
## 1. Setup & Initialisation

### Installation

```bash
pip install terndata.ecoplots
```

### Choosing a Client: Sync vs Async

| Client | Best for | `get_data()` call |
|---|---|---|
| **`EcoPlots`** | Notebooks, simple scripts | `df = ec.get_data(dformat="pd")` |
| **`AsyncEcoPlots`** | Production, concurrent I/O | `df = await ec.get_data(dformat="pd")` |

All other methods (`select`, `preview`, `summary`, discovery methods, widgets) are **identical** in both clients.

In [2]:
# ===================================================================
# Choose Your Client: Synchronous (simpler) or Asynchronous (advanced)
# ===================================================================
#
# Option 1: Synchronous - EcoPlots (no `await` needed)
# from terndata.ecoplots import EcoPlots
# ec = EcoPlots(mode="samples")
#
# Option 2: Asynchronous - AsyncEcoPlots (faster for large fetches)
from terndata.ecoplots import AsyncEcoPlots
ec = AsyncEcoPlots(mode="samples")
#
# Both clients behave identically in samples mode.
# The only difference: get_data() requires `await` in AsyncEcoPlots.
# ===================================================================

print(ec)

╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║                                                                              ║
║ Resolved Query Filters: 1                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


---

<a id="2-discovering-available-data"></a>
## 2. Discovering Available Data

Use discovery methods to explore what samples data exists **before** filtering or downloading. All discovery methods return a `pandas.DataFrame` and respect any filters you have already applied.

### 2.1 Material Sample Types

List all sample types registered in the system. Use the returned labels as values for `select(material_sample_type=...)`.

In [3]:
# Discover all available material sample types
sample_types = ec.get_material_sample_types()
print(f"Found {len(sample_types)} material sample types")
sample_types

Found 5 material sample types


,key,uri
0,Plant Tissue Sample,http://linked.data.gov.au/def/tern-cv/eee45444...
1,Plant Voucher Specimen,http://linked.data.gov.au/def/tern-cv/18317af1...
2,Soil Subsite Sample,http://linked.data.gov.au/def/tern-cv/f51d6ffa...
3,Soil Metagenomic Sample,http://linked.data.gov.au/def/tern-cv/f69d8f1e...
4,Soil Pit Sample,http://linked.data.gov.au/def/tern-cv/a49f0fcd...


### 2.2 Datasets

List datasets available within samples mode (fixed to *TERN Ecosystem Surveillance*).

In [4]:
# List datasets (always TERN Ecosystem Surveillance in samples mode)
datasets = ec.get_datasources()
print(f"Found {len(datasets)} dataset(s)")
datasets

Found 1 dataset(s)


,key,uri
0,TERN Ecosystem Surveillance,http://linked.data.gov.au/dataset/ausplots


### 2.3 Sites

Discover sites that have samples data. Apply a `material_sample_type` filter first to narrow the list.

In [5]:
# Optionally pre-filter before discovery — results will reflect the filter
ec.select(material_sample_type="Soil Pit Sample")

sites = ec.get_sites()
print(f"Found {len(sites)} sites with Soil Pit Samples")
sites.head(10)

Found 250 sites with Soil Pit Samples


,key,uri
0,SAAKAN0008,http://linked.data.gov.au/dataset/ausplots/sit...
1,SASMDD0001,http://linked.data.gov.au/dataset/ausplots/sit...
2,SASMDD0006,http://linked.data.gov.au/dataset/ausplots/sit...
3,SAAKAN0004,http://linked.data.gov.au/dataset/ausplots/sit...
4,SAAKAN0010,http://linked.data.gov.au/dataset/ausplots/sit...
5,SAAKAN0003,http://linked.data.gov.au/dataset/ausplots/sit...
6,SAAKAN0011,http://linked.data.gov.au/dataset/ausplots/sit...
7,WAAMUR0028,http://linked.data.gov.au/dataset/ausplots/sit...
8,QDAEIU0001,http://linked.data.gov.au/dataset/ausplots/sit...
9,SAAKAN0001,http://linked.data.gov.au/dataset/ausplots/sit...


### 2.4 Region Types & Regions

Discover the geographic region categories available and list regions within a chosen type.

In [6]:
# Step 1: see what region categories are available
region_types = ec.get_region_types()
print(f"Found {len(region_types)} region type(s)")
region_types

Found 7 region type(s)


,key,uri
0,Local Government Areas,https://linked.data.gov.au/dataset/asgsed3/LGA...
1,States and Territories,https://linked.data.gov.au/dataset/asgsed3/STE
2,NRM regions,https://linked.data.gov.au/dataset/ausnrm2023
3,IBRA7 Bioregions,https://linked.data.gov.au/dataset/ibra7
4,IBRA7 Subregions,https://linked.data.gov.au/dataset/ibra7/subre...
5,WWF ecoregions,https://linked.data.gov.au/dataset/wwf2011
6,Terrestrial CAPAD regions,https://linked.data.gov.au/dataset/auscapad2022


In [7]:
# Step 2: list regions within a chosen type
# Replace "bioregions" with any label returned above
regions = ec.get_regions("IBRA7 Bioregions")
print(f"Found {len(regions)} bioregions")
regions.head(10)

Found 61 bioregions


,key,uri
0,Murray Darling Depression,https://linked.data.gov.au/dataset/ibra7/MDD
1,Coolgardie,https://linked.data.gov.au/dataset/ibra7/COO
2,Flinders Lofty Block,https://linked.data.gov.au/dataset/ibra7/FLB
3,Stony Plains,https://linked.data.gov.au/dataset/ibra7/STP
4,Broken Hill Complex,https://linked.data.gov.au/dataset/ibra7/BHC
5,Kanmantoo,https://linked.data.gov.au/dataset/ibra7/KAN
6,Gulf Plains,https://linked.data.gov.au/dataset/ibra7/GUP
7,Mitchell Grass Downs,https://linked.data.gov.au/dataset/ibra7/MGD
8,Eyre Yorke Block,https://linked.data.gov.au/dataset/ibra7/EYB
9,Gulf Fall and Uplands,https://linked.data.gov.au/dataset/ibra7/GFU


### 2.5 Species Names

Discover species names linked to the current filters.
Only applicable when `material_sample_type` is `Plant Tissue Sample` or `Plant Voucher Specimen`.

In [8]:
# Switch to a plant sample type first
ec.clear()
ec.select(material_sample_type="Plant Voucher Specimen")

species = ec.get_speciesname()
print(f"Found {len(species)} species")
species.head(10)

Found 6109 species


,speciesname,count
0,Poaceae Barnhart,446
1,Salsola australis R.Br.,276
2,Sida fibulifera Lindl.,192
3,Enchylaena tomentosa R.Br. var. tomentosa,182
4,Aristida contorta F.Muell.,171
5,Enneapogon polyphyllus (Domin) N.T.Burb.,154
6,Dactyloctenium radulans (R.Br.) P.Beauv.,153
7,Cenchrus ciliaris L.,149
8,Tripogonella loliiformis (F.Muell.) P.M.Peters...,148
9,Sclerolaena diacantha (Nees) Benth.,146


---

<a id="3-applying-filters"></a>
## 3. Applying Filters

Filters are applied via `select()`. In samples mode the following facets are available:

| Facet | Type | Notes |
|---|---|---|
| `material_sample_type` | str | **Required before `get_data()`**; one type at a time |
| `site_id` | str / list | One or more site identifiers |
| `region_type` + `region` | str / list | Provide `region_type` together with or before `region` |
| `speciesname` | str / list | Plant tissue / voucher types only |
| `soil_subsite_id` | int / list[int] | Soil types only |
| `soil_depth_range` | [min, max] | Soil depth in metres |
| `has_images` | bool | Limit to samples with attached photographs |
| `spatial` | WKT / GeoJSON | Set by `select_spatial()` or passed directly |

In [9]:
# ----------------------------------------------------------------
# Start with a clean slate
# ----------------------------------------------------------------
ec.clear()

# ----------------------------------------------------------------
# Basic: select by material sample type and site
# ----------------------------------------------------------------
ec.select(
    material_sample_type="Soil Pit Sample",
    site_id="SATFLB0001",
)
print(ec)

╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Soil Pit Sample']                                ║
║   • site_id: ['SATFLB0001']                                                  ║
║                                                                              ║
║ Resolved Query Filters: 3                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


In [10]:
# ----------------------------------------------------------------
# Filter to samples within a geographic region
# ----------------------------------------------------------------
ec.clear()
ec.select(
    material_sample_type="Soil Pit Sample",
    region_type="IBRA7 bioregions",
    region="Flinders Lofty Block",
)
print(ec)


⚠️  Warning: Value 'IBRA7 bioregions' for facet 'region_type' corrected to 'IBRA7 Bioregions'.



╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Soil Pit Sample']                                ║
║   • region_type: ['IBRA7 Bioregions']                                        ║
║   • region: ['Flinders Lofty Block']                                         ║
║                                                                              ║
║ Resolved Query Filters: 4                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


In [11]:
# ----------------------------------------------------------------
# Soil depth range filter (metres)
# Restrict to samples collected between 0 m and 0.1 m depth
# ----------------------------------------------------------------
ec.select(soil_depth_range=[0.0, 0.1])
print(ec)

╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Soil Pit Sample']                                ║
║   • region_type: ['IBRA7 Bioregions']                                        ║
║   • region: ['Flinders Lofty Block']                                         ║
║   • soil_depth_range: [0.0, 0.1]                                             ║
║                                                                              ║
║ Resolved Query Filters: 5                                                    ║
╚═══════════════════════════

In [12]:
# ----------------------------------------------------------------
# Filter to plant vouchers that have images attached
# ----------------------------------------------------------------
ec.clear()
ec.select(
    material_sample_type="Plant Voucher Specimen",
    has_image=True,
)
print(ec)

╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Plant Voucher Specimen']                         ║
║   • has_image: True                                                          ║
║                                                                              ║
║ Resolved Query Filters: 3                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


In [13]:
# ----------------------------------------------------------------
# Multiple filters at once via method chaining
# ----------------------------------------------------------------
ec.clear()
ec.select(material_sample_type="Plant Tissue Sample") \
  .select(region_type="states and territories", region="Queensland")

print(ec)

╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Plant Tissue Sample']                            ║
║   • region_type: ['States and Territories']                                  ║
║   • region: ['Queensland']                                                   ║
║                                                                              ║
║ Resolved Query Filters: 4                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝



⚠️  Warning: Value 'states and territories' for facet 'region_type' corrected to 'States and Territories'.



---

<a id="4-inspecting--managing-filters"></a>
## 4. Inspecting & Managing Filters

These methods let you inspect what is currently active and remove individual
filter values without calling `clear()`.

| Method | Returns | Description |
|---|---|---|
| `get_filter(facet?)` | `dict` / `list` | Human-readable labels for all or one active facet |
| `get_api_query_filters(facet?)` | `dict` / `list` | Raw API URIs sent to Elasticsearch |
| `remove(**kwargs)` | `self` | Drop specific filter values (chainable) |
| `clear()` | `self` | Remove **all** active filters |

In [14]:
# ----------------------------------------------------------------
# Inspect all active filters (human-readable labels)
# ----------------------------------------------------------------
ec.clear()
ec.select(
    material_sample_type="Soil Pit Sample",
    region_type="IBRA7 Bioregions",
    region="Flinders Lofty Block",
)

filters = ec.get_filter()
print("Active filters (labels):")
for key, value in filters.items():
    print(f"  • {key}: {value}")


Active filters (labels):
  • dataset: ['TERN Ecosystem Surveillance']
  • material_sample_type: ['Soil Pit Sample']
  • region_type: ['IBRA7 Bioregions']
  • region: ['Flinders Lofty Block']


In [15]:
# View the raw API query representation (URIs sent to Elasticsearch)
query_filters = ec.get_api_query_filters()
print("Active API query filters (URIs):")
for key, value in query_filters.items():
    print(f"  • {key}: {value}")


Active API query filters (URIs):
  • dataset: ['http://linked.data.gov.au/dataset/ausplots']
  • material_sample_type: ['http://linked.data.gov.au/def/tern-cv/a49f0fcd-19f5-4098-ac2e-85a972286a43']
  • region_type: ['https://linked.data.gov.au/dataset/ibra7']
  • region: ['https://linked.data.gov.au/dataset/ibra7/FLB']


In [16]:
# ----------------------------------------------------------------
# Remove a specific filter value without clearing everything
# ----------------------------------------------------------------
ec.remove(region="Flinders Lofty Block")
print("After removing 'Flinders Lofty Block':")
print(ec)


After removing 'Flinders Lofty Block':
╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Soil Pit Sample']                                ║
║   • region_type: ['IBRA7 Bioregions']                                        ║
║                                                                              ║
║ Resolved Query Filters: 3                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


---

<a id="5-summary--preview"></a>
## 5. Summary & Preview

Use `summary()` to get a count of matching records **before** downloading, and
`preview()` to inspect a page of actual data — both are cheap operations and
respect the current filter state.

In [17]:
# Re-apply a representative filter set for the remaining examples
ec.clear()
ec.select(
    material_sample_type="Soil Pit Sample",
    region_type="IBRA7 Bioregions",
    region="Flinders Lofty Block",
)

# Get a count/summary of matching records
ec.summary()


,metric,count
0,sample_images,0
1,samples,232
2,site,31
3,site_visit,46
4,species,0


In [18]:
# Preview the first page of results — same columns as get_data(), no full download
preview_df = ec.preview()
print(f"Preview contains {len(preview_df)} records")
preview_df.head()


Preview contains 10 records


,dataset,feature_type,igsn,latitude,longitude,material_sample_type,sample_id,sample_name,site_id,site_id_label,site_visit_count,site_visit_id,soil_max_depth_metre,soil_min_depth_metre,used_procedure,visit_date,wkt_point
0,TERN Ecosystem Surveillance,soil profile,10.83032/saa000824,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000824,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.1,0.0,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
1,TERN Ecosystem Surveillance,soil profile,10.83032/saa000825,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000825,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.2,0.1,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
2,TERN Ecosystem Surveillance,soil profile,10.83032/saa000826,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000826,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.3,0.2,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
3,TERN Ecosystem Surveillance,soil profile,10.83032/saa000827,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000827,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.4,0.3,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
4,TERN Ecosystem Surveillance,soil profile,10.83032/saa000828,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000828,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.5,0.4,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)


---

<a id="6-retrieving-data"></a>
## 6. Retrieving Data

`get_data()` downloads the complete dataset matching the current filters.

| Parameter | Default | Options |
|---|---|---|
| `dformat` | `"gpd"` | `"gpd"` (GeoDataFrame), `"pd"` (DataFrame) |
| `allow_full_download` | `False` | Set `True` to skip the large-dataset prompt |

</br>

> **Note:**
> 1. Exactly **one** `material_sample_type` must be selected before calling `get_data()` in samples mode.
> 2. `GeoJSON` output is **not** currently available in samples mode and will raise an error if requested.

</br>

In [19]:
# ----------------------------------------------------------------
# Retrieve as a GeoDataFrame (default — includes geometry column)
# ----------------------------------------------------------------
# Synchronous:  df = ec.get_data(dformat="gpd")
# Asynchronous: df = await ec.get_data(dformat="gpd")  ← current client
# ----------------------------------------------------------------
df = await ec.get_data(dformat="gpd")
print(f"Retrieved {len(df)} records")
print(f"Columns: {list(df.columns)}")
df.head()


Retrieved 232 records
Columns: ['dataset', 'feature_type', 'igsn', 'latitude', 'longitude', 'material_sample_type', 'sample_id', 'sample_name', 'site_id', 'site_id_label', 'site_visit_count', 'site_visit_id', 'soil_max_depth_metre', 'soil_min_depth_metre', 'used_procedure', 'visit_date', 'wkt_point']


,dataset,feature_type,igsn,latitude,longitude,material_sample_type,sample_id,sample_name,site_id,site_id_label,site_visit_count,site_visit_id,soil_max_depth_metre,soil_min_depth_metre,used_procedure,visit_date,wkt_point
0,TERN Ecosystem Surveillance,soil profile,10.83032/saa000824,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000824,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.1,0.0,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
1,TERN Ecosystem Surveillance,soil profile,10.83032/saa000825,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000825,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.2,0.1,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
2,TERN Ecosystem Surveillance,soil profile,10.83032/saa000826,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000826,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.3,0.2,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
3,TERN Ecosystem Surveillance,soil profile,10.83032/saa000827,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000827,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.4,0.3,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)
4,TERN Ecosystem Surveillance,soil profile,10.83032/saa000828,-31.83779,138.3512,Soil Pit Sample,http://linked.data.gov.au/dataset/ausplots/sit...,SAA000828,http://linked.data.gov.au/dataset/ausplots/sit...,SAAFLB0008,1,20181010,0.5,0.4,Soil characterisation to 1 m+,2018-10-10T16:59:06,POINT(138.35119167 -31.83778611)


---

<a id="7-igsn-identifiers"></a>
## 7. IGSN Identifiers

Every sample is registered in the [International Geo Sample Number](https://www.igsn.org)
(IGSN) system and assigned a persistent DOI. The IGSN DOI landing page is the
authoritative record for a sample — it displays the officially registered
attributes: collection date, location, sample type, and parentage.

| Method | Description |
|---|---|
| `get_sample_igsn()` | Returns a DataFrame of `sample_name` → IGSN DOI pairs |
| `view_sample_igsn()` | Opens a widget: select a sample from the dropdown to load its DOI page in an iframe |
| `view_sample_igsn(igsn=...)` | Navigate directly to a known sample's DOI landing page |

</br>

> **Requires:** At least one `material_sample_type` must be selected.

</br>

In [20]:
# ----------------------------------------------------------------
# Retrieve IGSN DOIs for the currently filtered samples
# Each row: sample_name  |  igsn (10.60792/...)
# ----------------------------------------------------------------
ec.clear()
ec.select(material_sample_type="Soil Pit Sample")

igsn_df = ec.get_sample_igsn()
print(f"Found {len(igsn_df)} samples with IGSN identifiers")
igsn_df.head(10)


Found 4059 samples with IGSN identifiers


,sample_name,igsn
0,AUG-005000,10.60792/aug-005000
1,AUG-005001,10.60792/aug-005001
2,AUG-005002,10.60792/aug-005002
3,AUG-005003,10.60792/aug-005003
4,AUG-005004,10.60792/aug-005004
5,AUG-005005,10.60792/aug-005005
6,AUG-005006,10.60792/aug-005006
7,AUG-005007,10.60792/aug-005007
8,AUG-005008,10.60792/aug-005008
9,AUG-005009,10.60792/aug-005009


In [21]:
# Open the IGSN viewer widget
# Select a sample from the dropdown to load its DOI landing page in the iframe
ec.view_sample_igsn()

In [22]:
# Navigate directly to a known sample's landing page by providing its IGSN DOI
# Accepted formats:
#   "10.60792/aug-005009"
#   "doi.org/10.60792/aug-005009"
#   "https://doi.org/10.60792/aug-005009"
ec.view_sample_igsn(igsn="10.60792/aug-005009")

---

<a id="8-soil-analysis"></a>
## 8. Soil Analysis

These methods provide aggregated soil analysis views and are only meaningful
when a **soil** material sample type is selected.

| Method | Returns | Description |
|---|---|---|
| `get_soil_depth_range()` | `GeoDataFrame` | Sample counts aggregated by depth band |
| `get_soilpit()` | `DataFrame` | Soil pit distribution and counts per site |

</br>

> **Applicable sample types:** `Soil Pit Sample`, `Soil Subsite Sample`, `Soil Metagenomic Sample`

</br>

In [23]:
# Select a soil sample type to enable soil analysis methods
ec.clear()
ec.select(material_sample_type="Soil Pit Sample")
print(ec)


╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Soil Pit Sample']                                ║
║                                                                              ║
║ Resolved Query Filters: 2                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


In [24]:
# Aggregated depth-range summary
# Returns a GeoDataFrame — each row is a depth band with a sample count
# Note: depth bands are present in Soil Pit samples and Soil Subsite Samples only, not in Soil Metagenomic Samples
depth_df = ec.get_soil_depth_range()
print(f"Found {len(depth_df)} depth ranges")
depth_df.head(10)


Found 1 depth ranges


,overall_depth_min_meter,overall_depth_max_meter,min_depth_range,max_depth_range
0,0.0,40.0,0.0-30.0 m,0.0-40.0 m


In [25]:
# Soil pit distribution — number of soil pit samples per site
# Note: soil pit samples are present in Soil Metagenomic and Soil Subsite Sample types, not in Soil Pit Sample type.
ec.clear()
ec.select(material_sample_type="Soil Subsite Sample")
soilpit_df = ec.get_soilpit()
print(f"Found {len(soilpit_df)} soil pit entries")
soilpit_df.head(10)

Found 9 soil pit entries


,soilpit,counts
0,2,2191
1,1,2185
2,3,2182
3,4,2178
4,5,2170
5,7,2154
6,6,2145
7,9,2136
8,8,2133


---

<a id="9-species-distribution"></a>
## 9. Species Distribution

`get_speciesname()` discovers the species names linked to the current filters.
It is only available when `material_sample_type` is `Plant Tissue Sample` or
`Plant Voucher Specimen`. Use a returned name directly in `select(speciesname=...)`
to narrow your query to a particular taxon.

In [26]:
# ----------------------------------------------------------------
# Discover species names for plant sample types
# ----------------------------------------------------------------
ec.clear()
ec.select(material_sample_type="Plant Voucher Specimen")

species_df = ec.get_speciesname()
print(f"Found {len(species_df)} species")
species_df.head(15)


Found 6109 species


,speciesname,count
0,Poaceae Barnhart,446
1,Salsola australis R.Br.,276
2,Sida fibulifera Lindl.,192
3,Enchylaena tomentosa R.Br. var. tomentosa,182
4,Aristida contorta F.Muell.,171
5,Enneapogon polyphyllus (Domin) N.T.Burb.,154
6,Dactyloctenium radulans (R.Br.) P.Beauv.,153
7,Cenchrus ciliaris L.,149
8,Tripogonella loliiformis (F.Muell.) P.M.Peters...,148
9,Sclerolaena diacantha (Nees) Benth.,146


In [27]:
# Use a species name as a filter
# Replace the value below with a name returned by get_speciesname()
ec.select(speciesname="Poaceae Barnhart")
print(ec)


╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Plant Voucher Specimen']                         ║
║   • speciesname: ['Poaceae Barnhart']                                        ║
║                                                                              ║
║ Resolved Query Filters: 3                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝


---

<a id="10-sample-image-viewer"></a>
## 10. Sample Image Viewer

`view_sample_images()` opens an interactive viewer for samples that have
photographs attached. Each image may be available in multiple resolutions —
use the resolution selector in the widget to switch between them.

The method can:
- fetch the data itself (no arguments required), or
- accept a pre-fetched `pandas.DataFrame` via the `data=` parameter to skip a
  second round-trip to the API.

> **Recommended:** Add `has_images=True` to your filters — it guarantees the
> viewer has images to display.

In [29]:
# ----------------------------------------------------------------
# Select plant vouchers that have images and open the viewer
# ----------------------------------------------------------------
ec.clear()
ec.select(
    material_sample_type="Plant Voucher Specimen",
    has_image=True,
)
# The widget fetches data automatically and displays a navigable image grid
ec.view_sample_images()


In [ ]:
# ----------------------------------------------------------------
# Alternatively, pass a pre-fetched DataFrame to avoid a second API call
# (useful when you already have the data from get_data())
# ----------------------------------------------------------------
df_plants = await ec.get_data(dformat="pd")
ec.view_sample_images(data=df_plants)


---

<a id="11-spatial-filter-widget"></a>
## 11. Spatial Filter Widget

`select_spatial()` opens an interactive map widget. Draw a rectangle or polygon
over your area of interest, then click **Confirm Selection** — the client
converts the drawing to a WKT polygon and stores it as the `spatial` filter.

The filter can also be set directly via `select(spatial=<wkt>)` if you already
have a WKT string (e.g. copied from GIS software or another notebook session).

In [30]:
# Open the spatial selector widget
# Draw a shape on the map and click "Confirm Selection" to apply the filter
ec.clear()
ec.select(material_sample_type="Soil Pit Sample")
ec.select_spatial()


In [31]:
# Inspect the spatial filter that was applied by the widget
print(ec)


╔══════════════════════════════════════════════════════════════════════════════╗
║ AsyncEcoPlots Samples                                                        ║
║ Version: 0.0.7-beta                                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Active Filters:                                                              ║
║   • dataset: ['TERN Ecosystem Surveillance']                                 ║
║   • material_sample_type: ['Soil Pit Sample']                                ║
║   • spatial: POLYGON ((121.201172 -35.532226, 121.201172 -30.600094, 114.521 ║
║     484 -30.600094, 114.521484 -35.532226, 121.201172 -35.532226))           ║
║                                                                              ║
║ Resolved Query Filters: 3                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝
